# 11주차 — Sensors 팔레트: ideal → noisy 의 순간 ★

```
[Sensors] → [Estimation] → [Navigation] → [Control] → [Allocation] → [Plant]
 ▲ 이번 주
```

목표: **완벽한 센서라는 거짓말.** 지금까지 제어기는 플랜트의 참값을 읽었다.
실제 기체에는 참값이 없다 — IMU 가 있을 뿐이다.

## 이론 요약
- IMU = 자이로(각속도) + 가속도계(비력). 레벨 정지에서 가속도계는 [0,0,−g]
- 노이즈 모델 (ICM-42688 급 datasheet): bias + 백색잡음(noise density ×
  √대역폭) + 샘플레이트 제한
- **fidelity 규약** (ARCHITECTURE §2): ideal ↔ noisy ↔ gazebo ↔ hardware —
  포트는 절대 불변. 다이어그램은 하나.
- 시드 고정: 채점/회귀 재현성 (같은 seed = 같은 잡음열)

In [ ]:
# 뷰어 열기 — 뼈대 모델을 브라우저에서 편집한다.
# 완성하면 뷰어의 '저장' → week11_model.json 이 다운로드된다.
# 그 파일을 이 폴더의 같은 이름 파일에 **덮어쓰면 제출 완료** (채점 셀로 확인).
import sys, webbrowser; sys.path.insert(0, '../..')
from loopsym import load_json, render_html
p = render_html(load_json('week11_model.json'), 'week11_viewer.html',
                title='11주차', t_end=8.0, dt=0.004,
                save_name='week11_model.json')
webbrowser.open('file://' + p)

## 실습 — 이 한 번의 클릭
① 10주차 다이어그램에 IMU 블록 삽입: 플랜트 q/ω/vel → IMU 입력,
   rate_ctrl 의 각속도 피드백을 참값 대신 **IMU gyro 출력**으로 재배선.
② IMU 더블클릭 → fidelity 를 `ideal` → `noisy` 로. **기체가 흔들리기
   시작하는 것을 목격하라.** 이 클릭이 상태추정이라는 분야가 존재하는 이유다.
   (gyro_bias=0.02, gyro_nd=1e-3 으로 설정 — 채점 기준)

In [ ]:
# 노트북에서 바로 실행해 보기 (뷰어 없이) — 저장한 모델을 로드해 시뮬
import sys; sys.path.insert(0, '../..')
from loopsym import load_json, Simulator
bd = load_json('week11_model.json')
sim = Simulator(bd); sim.run(8.0, 0.004)
import matplotlib.pyplot as plt
for b in sim.bd.blocks:
    if type(b).__name__ == 'Scope':
        plt.figure(figsize=(8,2.6)); plt.title(b.name); plt.grid(True)
        for i, lab in enumerate(b.labels): plt.plot(b.t, b.data[i], label=lab)
        plt.legend()

## 과제
1. week11_model.json 완성 (채점기: IMU 구조 + noisy + 미션 유지)
2. 경계 리포트: gyro_nd 를 1e-3 → 1e-2 → 5e-2 로 키우며 어느 수준부터
   미션이 실패하는지 경계를 찾아라 (seed 고정 후 스윕).

In [ ]:
# ✅ 채점 — 전부 초록색이 되면 이번 주 완료!
!python -m pytest test_week11.py -q